# TP4. Matrices et systèmes linéaires

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp4-matrices-gauss/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la transposée

Écrivez `transposee(A)` qui échange lignes et colonnes.

**Résultat attendu :** la transposée de `[[1, 2, 3], [4, 5, 6]]` est
`[[1, 4], [2, 5], [3, 6]]`.


In [1]:
def transposee(A):
    """Transposée de la matrice A, donnée comme liste de lignes."""
    lignes = len(A)
    colonnes = len(A[0])
    return [[A[i][j] for i in range(lignes)] for j in range(colonnes)]

print(transposee([[1, 2, 3], [4, 5, 6]]))



[[1, 4], [2, 5], [3, 6]]


## Exercice 2.1 : produit à la main

Écrivez `produit(A, B)`, et vérifiez sur un exemple que vous savez calculer de tête.

**Résultat attendu :** multiplier une matrice par l'identité la laisse inchangée.


In [2]:
def produit(A, B):
    """Produit matriciel A x B, en Python pur."""
    n = len(A)
    p = len(A[0])
    if len(B) != p:
        raise ValueError("dimensions incompatibles")
    q = len(B[0])
    
    # Création d'une matrice vide C de taille n x q
    C = [[0] * q for _ in range(n)]
    
    # Calcul du produit
    for i in range(n):
        for j in range(q):
            somme = 0
            for k in range(p):
                somme += A[i][k] * B[k][j]
            C[i][j] = somme
    return C

identite = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
print("A x Id == A :", produit(A, identite) == A)
print("Exemple 2x2 :", produit([[1, 2], [3, 4]], [[0, 1], [1, 0]]))


A x Id == A : True
Exemple 2x2 : [[2, 1], [4, 3]]


## Exercice 2.2 : mesurer le coût cubique

Mesurez le temps de `produit` sur deux matrices carrées aléatoires de taille 40, 80
puis 120. Que devient le temps quand la taille double ?

**Résultat attendu :** le temps est multiplié par environ 8 quand la taille double.


In [3]:
import random
import time

random.seed(42)

def matrice_aleatoire(n):
    return [[random.random() for _ in range(n)] for _ in range(n)]

for n in (40, 80, 120):
    A = matrice_aleatoire(n)
    B = matrice_aleatoire(n)
    
    debut = time.perf_counter()
    produit(A, B)
    duree = time.perf_counter() - debut
    
    print(f"n = {n:>3} : {duree:.4f} s")

n =  40 : 0.0079 s
n =  80 : 0.0574 s
n = 120 : 0.1976 s


## Exercice 3.1 : le rapport de temps

Reprenez la mesure de l'exercice 2.2 pour `n = 120`, et comparez au produit numpy des
mêmes matrices.

**Résultat attendu :** numpy est plus rapide de deux à trois ordres de grandeur.


In [7]:
import time
import random
import numpy as np

random.seed(42)
n = 120
A = matrice_aleatoire(n)
B = matrice_aleatoire(n)

# Mesure Python pur
debut = time.perf_counter()
produit(A, B)
t_python = time.perf_counter() - debut

# Mesure NumPy
An = np.array(A)
Bn = np.array(B)
debut = time.perf_counter()
An @ Bn
t_numpy = time.perf_counter() - debut

print(f"python pur {t_python:.4f} s")
print(f"numpy      {t_numpy:.6f} s")
print("rapport :", round(t_python / max(t_numpy, 1e-9)))



python pur 0.1969 s
numpy      0.001655 s
rapport : 119


## Exercice 4.1 : élimination et remontée

Écrivez `resoudre(A, b)` qui résout le système par le pivot de Gauss avec pivot
partiel, et testez-la sur le système de l'étape 1.

**Résultat attendu :** la solution est $(3,\, -1,\, 0{,}5)$, et $A x$
redonne bien $b$.


In [4]:
def resoudre(A, b, tolerance=1e-14):
    n = len(A)
    # Matrice augmentée
    M = [list(A[i]) + [b[i]] for i in range(n)]

    for k in range(n):
        # Recherche du pivot partiel
        meilleure = max(range(k, n), key=lambda i: abs(M[i][k]))
        if abs(M[meilleure][k]) <= tolerance:
            raise ValueError("systeme singulier ou mal pose")
            
        # Échange de ligne
        M[k], M[meilleure] = M[meilleure], M[k]

        # Élimination
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n + 1):
                M[i][j] -= facteur * M[k][j]

    # Remontée
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        somme = M[i][n]
        for j in range(i + 1, n):
            somme -= M[i][j] * x[j]
        x[i] = somme / M[i][i]
        
    return x

A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
b = [2, -1, 8]
x = resoudre(A, b)
print("Solution x :", [round(v, 10) for v in x])

verif = [sum(A[i][j] * x[j] for j in range(3)) for i in range(3)]
print("Vérification A x :", [round(v, 10) for v in verif])


Solution x : [3.0, -1.0, 0.5]
Vérification A x : [2.0, -1.0, 8.0]


## Exercice 4.2 : confronter à numpy

Comparez votre solution à celle de `numpy.linalg.solve` sur un système aléatoire de
taille 50.

**Résultat attendu :** les deux solutions coïncident à $10^{-9}$ près.


In [8]:
import numpy as np
import random

random.seed(7)
n = 50
A = matrice_aleatoire(n)
b = [random.random() for _ in range(n)]

mienne = resoudre(A, b)
celle_de_numpy = np.linalg.solve(np.array(A), np.array(b))

ecart = max(abs(mienne[i] - celle_de_numpy[i]) for i in range(n))
print("ecart maximal :", ecart)
print("Précision valide :", ecart < 1e-9)


ecart maximal : 4.956035581926699e-13
Précision valide : True


## Exercice 5.1 : lire le résultat

Exécutez le bloc ci-dessus et commentez. Quelle méthode donne la solution la plus
proche de la vraie ? Que vaut le conditionnement, et qu'est-ce que cela signifie ?


In [9]:
import numpy as np

def hilbert(n):
    return np.array([[1.0 / (i + j + 1) for j in range(n)] for i in range(n)])

n = 12
H = hilbert(n)
x_vrai = np.ones(n)          
b = H @ x_vrai               

x_solve = np.linalg.solve(H, b)
x_inverse = np.linalg.inv(H) @ b

print("erreur avec solve   :", float(np.max(np.abs(x_solve - x_vrai))))
print("erreur avec inv @ b :", float(np.max(np.abs(x_inverse - x_vrai))))
print("conditionnement     : {:.3e}".format(float(np.linalg.cond(H))))


erreur avec solve   : 0.38833107451177806
erreur avec inv @ b : 20.7099609375
conditionnement     : 1.803e+16


## Exercice 6.1 : composer deux transformations

Vérifiez numériquement que faire tourner de 30 degrés puis de 60 degrés revient à
faire tourner de 90 degrés, autrement dit que
$R(60°) \cdot R(30°) = R(90°)$.


In [10]:
import numpy as np

def rotation(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])

gauche = rotation(np.pi / 3) @ rotation(np.pi / 6)
droite = rotation(np.pi / 2)

print("Matrices égales ?", np.allclose(gauche, droite))
print(np.round(droite, 10))

Matrices égales ? True
[[ 0. -1.]
 [ 1.  0.]]


## Exercice 7.1 : déterminant par Gauss

Écrivez `determinant(A)` en reprenant l'élimination de `resoudre`, sans second membre :
multipliez les pivots successifs, et changez le signe à chaque échange de lignes.
Comparez à `np.linalg.det`.

**Résultat attendu :** le déterminant de la matrice de l'étape 1 vaut `-2`, et celui
de `[[1, 2], [2, 4]]` vaut `0`.


In [11]:
import numpy as np

def determinant(A):
    """Déterminant de A par élimination de Gauss."""
    n = len(A)
    M = [list(ligne) for ligne in A]
    det = 1.0
    for k in range(n):
        meilleure = max(range(k, n), key=lambda i: abs(M[i][k]))
        if abs(M[meilleure][k]) < 1e-14:
            return 0.0
        if meilleure != k:
            M[k], M[meilleure] = M[meilleure], M[k]
            det = -det
        det *= M[k][k]
        
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] -= facteur * M[k][j]
    return det

A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
print("Det Gauss :", determinant(A))
print("Det NumPy :", float(np.linalg.det(np.array(A, dtype=float))))
print("Det matrice singulière :", determinant([[1, 2], [2, 4]]))



Det Gauss : -2.000000000000001
Det NumPy : -2.0000000000000013
Det matrice singulière : 0.0


## Exercice 8.1 : Hilbert vaincue

Construisez la matrice de Hilbert de taille 12 avec des `Fraction(1, i + j + 1)`, le
second membre `b` comme somme des lignes, pour que la solution soit le vecteur de 1,
puis appelez **votre** fonction `resoudre` de l'étape 4 dessus. Le premier essai échoue
avec « système singulier ». Comprenez pourquoi avant de lire le corrigé, puis corrigez.

**Résultat attendu :** exactement `[1, 1, ..., 1]`, douze fois `Fraction(1, 1)`, là
où `solve` se trompait de 0,7.


In [12]:
from fractions import Fraction

def hilbert_exacte(n):
    return [[Fraction(1, i + j + 1) for j in range(n)] for i in range(n)]

n = 12
H_exacte = hilbert_exacte(n)
b_exact = [sum(ligne) for ligne in H_exacte]

try:
    resoudre(H_exacte, b_exact) # Échoue à cause de la tolérance 1e-14
except ValueError as erreur:
    print("premier essai :", erreur)

# En désactivant la tolérance :
x_exact = resoudre(H_exacte, b_exact, tolerance=0)
print("Tous égaux à 1 ?", all(v == 1 for v in x_exact))


premier essai : systeme singulier ou mal pose
Tous égaux à 1 ? True


## Exercice 9.1 : l'ordre compte

Écrivez `translation(tx, ty)` et `rotation_h(angle)` qui renvoient des matrices
$3 \times 3$. Appliquez au point $(1, 0)$ la translation de $(3, 1)$ suivie
de la rotation de 90°, puis la rotation suivie de la translation.

**Résultat attendu :** $(-1, 4)$ dans le premier cas, $(3, 2)$ dans le second.
Les deux résultats diffèrent.


In [13]:
import numpy as np

def translation(tx, ty):
    return np.array([[1.0, 0.0, tx],
                     [0.0, 1.0, ty],
                     [0.0, 0.0, 1.0]])

def rotation_h(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s, 0.0],
                     [s, c, 0.0],
                     [0.0, 0.0, 1.0]])

point = np.array([1.0, 0.0, 1.0])
T = translation(3, 1)
R = rotation_h(np.pi / 2)

print("Translation puis rotation :", np.round(R @ T @ point, 10))   
print("Rotation puis translation :", np.round(T @ R @ point, 10))


Translation puis rotation : [-1.  4.  1.]
Rotation puis translation : [3. 2. 1.]
